In [ ]:
# fine_tune_hybrid.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import pickle
from tqdm import tqdm

# ===== CONFIG =====
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = "data/train"
SSL_ENCODER_PATH = "ssl_encoder.pth"
FSL_ENCODER_PATH = "fsl_encoder.pth"
OLD_HYBRID_PATH = "hybrid_model.pth"
SAVE_MODEL_PATH = "hybrid_finetuned.pth"
SAVE_PKL_PATH = "hybrid_features_finetuned.pkl"
EPOCHS = 10
BATCH_SIZE = 2
LR = 1e-5  # low LR for fine-tuning
NUM_CLASSES = 2  # adjust to your dataset

# ===== DATASET =====
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
train_data = datasets.ImageFolder(root=DATA_ROOT, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
print(f"📂 Loaded dataset with {len(train_data)} samples across {len(train_data.classes)} classes.")

# ===== LOAD ENCODERS =====
def load_encoder(path):
    encoder = models.resnet18(weights=None)
    encoder = nn.Sequential(*list(encoder.children())[:-1])
    state = torch.load(path, map_location=DEVICE)
    try:
        encoder.load_state_dict(state, strict=False)
        print(f"✅ Loaded encoder weights from {path}")
    except Exception as e:
        print(f"⚠️ Partial load for {path}: {e}")
    return encoder.to(DEVICE)

ssl_encoder = load_encoder(SSL_ENCODER_PATH)
fsl_encoder = load_encoder(FSL_ENCODER_PATH)

# ===== HYBRID MODEL =====
class HybridModel(nn.Module):
    def __init__(self, ssl_encoder, fsl_encoder, num_classes):
        super().__init__()
        self.ssl_encoder = ssl_encoder
        self.fsl_encoder = fsl_encoder
        self.fc = nn.Sequential(
            nn.Linear(512*2, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        ssl_feat = self.ssl_encoder(x).flatten(1)
        fsl_feat = self.fsl_encoder(x).flatten(1)
        fused = torch.cat([ssl_feat, fsl_feat], dim=1)
        out = self.fc(fused)
        return out, fused

hybrid_model = HybridModel(ssl_encoder, fsl_encoder, NUM_CLASSES).to(DEVICE)

# ===== LOAD OLD HYBRID MODEL =====
hybrid_model.load_state_dict(torch.load(OLD_HYBRID_PATH, map_location=DEVICE))
print("✅ Loaded old hybrid_model.pth")

# ===== UNFREEZE LAYERS FOR FINE-TUNING =====
for param in hybrid_model.parameters():
    param.requires_grad = False

# Unfreeze last block of each encoder
for param in hybrid_model.ssl_encoder[7].parameters():
    param.requires_grad = True
for param in hybrid_model.fsl_encoder[7].parameters():
    param.requires_grad = True
# Also train FC head
for param in hybrid_model.fc.parameters():
    param.requires_grad = True

# ===== OPTIMIZER =====
optimizer = optim.Adam(filter(lambda p: p.requires_grad, hybrid_model.parameters()), lr=LR)

# ===== TRAINING =====
hybrid_model.train()
for epoch in range(EPOCHS):
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs, _ = hybrid_model(imgs)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Acc: {acc*100:.2f}%")

# ===== SAVE FINE-TUNED MODEL =====
torch.save(hybrid_model.state_dict(), SAVE_MODEL_PATH)
print(f"✅ Fine-tuned hybrid model saved as {SAVE_MODEL_PATH}")

# ===== EXTRACT FEATURES & SAVE PKL =====
hybrid_model.eval()
all_feats, all_labels, all_preds = [], [], []
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        outputs, feats = hybrid_model(imgs)
        preds = outputs.argmax(1).cpu().numpy()
        all_feats.append(feats.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(preds)

all_feats = np.concatenate(all_feats)
all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)
acc = (all_preds == all_labels).mean()

with open(SAVE_PKL_PATH, "wb") as f:
    pickle.dump({
        "features": all_feats,
        "labels": all_labels,
        "predictions": all_preds,
        "accuracy": acc
    }, f)
print(f"✅ Features and metrics saved to {SAVE_PKL_PATH} (Acc: {acc*100:.2f}%)")


📂 Loaded dataset with 1000 samples across 2 classes.
✅ Loaded encoder weights from ssl_encoder.pth
✅ Loaded encoder weights from fsl_encoder.pth
✅ Loaded old hybrid_model.pth


Epoch 1/10:   0%|          | 0/500 [00:00<?, ?it/s]c:\Users\Dabie\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Epoch 1/10: 100%|██████████| 500/500 [17:45<00:00,  2.13s/it]


Epoch 1/10 | Loss: 0.5962 | Acc: 68.40%


Epoch 2/10:  59%|█████▉    | 297/500 [15:05<12:34,  3.72s/it]